# FastSpeech 2 Text-to-Speech (English & Telugu) — Run 2 Recovery Pipeline (`R0`–`R5`)

### Recovery Design & Safety Guarantees
- **Root Cause Addressed (`b7718dd+`):** MFA 2.2.17 writes interior pauses as empty intervals `""` in `.TextGrid` files. In Run 1, `tgt.io.read_textgrid()` defaulted to `include_empty_intervals=False`, dropping interior pauses and shifting subsequent phonemes onto earlier audio frames.
- **Immutable Run 1 Evidence (`R0`):** Run 1 archives (`English_preprocessed.tar.gz`, `Telugu_preprocessed.tar.gz`), checkpoints, logs, and audio outputs in `/content/drive/MyDrive/FastSpeech2/` remain untouched. Run 2 writes exclusively under `/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2/`. Local git checkouts are inspected safely without destructive `git reset --hard`.
- **Verified Alignment Reuse & Rebuild (`R1`):** Extracts only `.TextGrid` files, `lexicon`, and Run 1 manifests/stats into staging, regenerates `raw_data` waveforms (`seed=42`), verifies primary/reserve manifests and waveform-to-TextGrid durations, runs hardened `preprocess.py`, enforces the ~5h train / ~1h val targets and train-only `stats.json` bounds, and only then writes `*_preprocessed_v2.tar.gz`.
- **Staged Quality Gates (`R2` -> `R3` -> `R4` Pilot -> `R4` Full -> `R5`):**
  - **R2:** Isolated per-language subprocess vocoder & frontend gate (real-mel $\rightarrow$ Universal HiFi-GAN reconstructions paired with original audio for English and both Telugu voices `te_female`, `te_male`, plus frontend checks on `E1–E4` and `T1–T4`).
  - **R3:** 64-clip, 2,000-step overfit diagnostic comparing **reference-conditioned** vs. **free-running** (predicted duration, pitch, energy) synthesis.
  - **R4:** Defaults to a **5,000-step full-data pilot** (`PILOT_OR_TOTAL_STEP = 5000`) with validation every 1,000 steps and a free-running checkpoint listening sweep (**R4-C**). Full training (`60,000` steps) requires explicitly setting `ALLOW_FULL_PRODUCTION_RUN = True` after reviewing R1–R3 and pilot audio.
  - **R5:** Synthesizes all 10 untouched test sentences (`E1–E4`, `T1–T4`), verifies `variance_predictions_summary.json`, 10 WAVs, 30 variance plots, and 4 loss curves, and packages `Results_Deliverables_v2.zip`.

## Section R0: Mount Drive, Safely Update/Verify Repo Checkout, Inventory Run 1 & Initialize Run 2 (`pause_fix_v2`)

In [ ]:
import os, sys, json, glob, shutil, hashlib, subprocess
from datetime import datetime, timezone
os.environ["MPLBACKEND"] = "Agg"

from google.colab import drive
drive.mount("/content/drive")

RUN1_ROOT = "/content/drive/MyDrive/FastSpeech2"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
REPO_URL = "https://github.com/SandeshAwate6055/FastSpeech2-English-Telugu.git"
REPO_DIR = "/content/FastSpeech2-English-Telugu"

# Create isolated Run 2 directory hierarchy
for sub in [
    "diagnostics",
    "diagnostics/vocoder_gate",
    "diagnostics/overfit64",
    "diagnostics/ckpt_sweep",
    "English/checkpoints",
    "English/logs",
    "Telugu/checkpoints",
    "Telugu/logs",
    "Results/English",
    "Results/Telugu",
]:
    os.makedirs(os.path.join(RUN2_ROOT, sub), exist_ok=True)

# Clone or safely fast-forward repository without discarding uncommitted local work
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    dirty_status = subprocess.check_output(["git", "-C", REPO_DIR, "status", "--porcelain"], text=True).strip()
    if dirty_status:
        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        patch_path = os.path.join(RUN2_ROOT, "diagnostics", f"dirty_checkout_{ts}.patch")
        status_path = os.path.join(RUN2_ROOT, "diagnostics", f"dirty_status_{ts}.txt")
        with open(status_path, "w", encoding="utf-8") as f:
            f.write(dirty_status + "\n")
        diff_txt = subprocess.check_output(["git", "-C", REPO_DIR, "diff"], text=True)
        with open(patch_path, "w", encoding="utf-8") as f:
            f.write(diff_txt)
        raise RuntimeError(
            f"Colab checkout at {REPO_DIR} has uncommitted changes! "
            f"Saved status to {status_path} and diff to {patch_path}. "
            f"Inspect or stash/clean before pulling."
        )
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)

git_commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print(f"Verified clean repository checkout at commit: {git_commit}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
subprocess.run([
    sys.executable, "-c",
    "import nltk; nltk.download('averaged_perceptron_tagger', quiet=True); "
    "nltk.download('averaged_perceptron_tagger_eng', quiet=True); nltk.download('cmudict', quiet=True)"
], check=True)

# Ensure Universal HiFi-GAN vocoder weights exist in both language folders
for lang in ["English", "Telugu"]:
    hifi_ckpt = os.path.join(REPO_DIR, lang, "hifigan", "generator_universal.pth.tar")
    if not os.path.exists(hifi_ckpt):
        if not os.path.exists("/tmp/ming_fs2"):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ming024/FastSpeech2.git", "/tmp/ming_fs2"], check=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/ming_fs2/hifigan/generator_universal.pth.tar.zip", "-d", os.path.join(REPO_DIR, lang, "hifigan")], check=True)
        print(f"Extracted generator_universal.pth.tar for {lang}")

# Inventory Run 1 artifacts (read-only)
run1_inventory = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit,
    "run1_root": RUN1_ROOT,
    "run2_root": RUN2_ROOT,
    "archives": {},
    "languages": {},
}
for lang in ["English", "Telugu"]:
    arc = os.path.join(RUN1_ROOT, f"{lang}_preprocessed.tar.gz")
    run1_inventory["archives"][lang] = {
        "path": arc,
        "exists": os.path.exists(arc),
        "size_mb": round(os.path.getsize(arc) / (1024 * 1024), 2) if os.path.exists(arc) else 0,
    }
    ckpts = sorted(glob.glob(os.path.join(RUN1_ROOT, lang, "checkpoints", "*.pth.tar")))
    wavs = sorted(glob.glob(os.path.join(RUN1_ROOT, "Results", lang, "test_generated_audios", "*.wav")))
    run1_inventory["languages"][lang] = {
        "checkpoints": [os.path.basename(c) for c in ckpts],
        "train_log_exists": os.path.exists(os.path.join(RUN1_ROOT, lang, "logs", "train_losses.csv")),
        "val_log_exists": os.path.exists(os.path.join(RUN1_ROOT, lang, "logs", "val_losses.csv")),
        "run1_test_wavs": [os.path.basename(w) for w in wavs],
    }

inv_path = os.path.join(RUN2_ROOT, "diagnostics", "run1_inventory.json")
with open(inv_path, "w", encoding="utf-8") as f:
    json.dump(run1_inventory, f, indent=2)

print("\n=== Run 1 Preservation & Inventory Summary ===")
print(json.dumps(run1_inventory, indent=2))


## Section R1: Alignment-Only Restoration, Raw Waveform Regeneration (`seed=42`), Manifest/Timeline Verification & Feature Rebuild
- **Cell R1-A:** Restores only `.TextGrid` + `lexicon` from Run 1 into staging, regenerates `raw_data` waveforms (`seed=42`), parses `utt_id|spk_id|dur|sha1|text` accurately, requires primary manifests (`train_filelist.txt`, `val_filelist.txt`) to exist and match 100% on IDs/speakers/durations/SHA-1s, checks reserve manifests (`reserve_train_filelist.txt`, `reserve_val_filelist.txt`) or `data_stats.json` reserve counts, and verifies that every aligned `.TextGrid` matches a regenerated `.wav` of equal duration before running `preprocess.py`.
- **Cell R1-B:** Runs the mandatory **R1 Integrity, Split-Hour (`~5h` train / `~1h` val), Train-Only `stats.json` & Pause-Recovery Audit**, and only archives `English_preprocessed_v2.tar.gz` and `Telugu_preprocessed_v2.tar.gz` **after** all checks pass.

In [ ]:
# Cell R1-A: Restore Verified Alignments Only, Regenerate Raw Audio, Verify Manifests & Run Fixed Preprocessor
import os, sys, json, glob, shutil, subprocess
from getpass import getpass
from scipy.io import wavfile
import tgt

RUN1_ROOT = "/content/drive/MyDrive/FastSpeech2"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
REPO_DIR = "/content/FastSpeech2-English-Telugu"
STAGING_DIR = "/content/run1_staging"

# Set REUSE_VALIDATED_V2_ARCHIVE = True only if you already built & audited v2 archives in this run and are restoring on a new GPU runtime.
# Even when True, Cell R1-B below MUST still be executed to re-verify all split, hour, and feature invariants.
REUSE_VALIDATED_V2_ARCHIVE = False

def parse_manifest(path):
    """Parse prepare_data.py manifest format: utt_id|spk_id|dur|sha1|text"""
    rows = {}
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.rstrip("\r\n")
            if not line:
                continue
            parts = line.split("|", 4)
            if len(parts) != 5:
                raise ValueError(f"Malformed manifest line {line_num} in {path}: {line!r}")
            utt_id, spk_id, dur_str, sha1, text = parts
            rows[utt_id] = {
                "spk": spk_id,
                "dur": float(dur_str),
                "sha1": sha1,
                "text": text,
            }
    return rows

def compare_manifests(lang, split_name, old_m, new_m, required=True):
    if old_m is None:
        if required:
            raise RuntimeError(f"[{lang}] Required Run 1 manifest {split_name} is missing from staging archive!")
        print(f"   [{split_name}] Not archived in Run 1 tarball (will verify via data_stats.json & TextGrid-to-WAV checks); new count={len(new_m)}")
        return
    if new_m is None:
        raise RuntimeError(f"[{lang}] Regenerated manifest {split_name} is missing!")
    old_ids = set(old_m.keys())
    new_ids = set(new_m.keys())
    only_old = len(old_ids - new_ids)
    only_new = len(new_ids - old_ids)
    common = old_ids & new_ids
    spk_err = sum(1 for k in common if old_m[k]["spk"] != new_m[k]["spk"])
    sha_err = sum(1 for k in common if old_m[k]["sha1"] != new_m[k]["sha1"])
    dur_err = sum(1 for k in common if abs(old_m[k]["dur"] - new_m[k]["dur"]) > 1e-2)
    txt_err = sum(1 for k in common if old_m[k]["text"] != new_m[k]["text"])
    print(
        f"   Manifest check [{split_name}]: old={len(old_ids)}, new={len(new_ids)}, "
        f"only_old={only_old}, only_new={only_new}, spk_err={spk_err}, "
        f"sha1_err={sha_err}, dur_err={dur_err}, txt_err={txt_err}"
    )
    if only_old > 0 or only_new > 0 or spk_err > 0 or sha_err > 0 or dur_err > 0 or txt_err > 0:
        raise RuntimeError(f"[{lang}] Manifest mismatch in {split_name}! Refusing to apply Run 1 TextGrids to mismatched data.")

for lang in ["English", "Telugu"]:
    lang_dir = os.path.join(REPO_DIR, lang)
    v2_arc = os.path.join(RUN2_ROOT, f"{lang}_preprocessed_v2.tar.gz")
    audit_ok_path = os.path.join(RUN2_ROOT, "diagnostics", "r1_integrity_audit.json")

    if REUSE_VALIDATED_V2_ARCHIVE and os.path.exists(v2_arc) and os.path.exists(audit_ok_path):
        print(f"\n[{lang}] Restoring previously audited Run 2 archive {v2_arc} into {lang_dir} ...")
        for stale in ["preprocessed_data", "raw_data"]:
            p = os.path.join(lang_dir, stale)
            if os.path.exists(p):
                shutil.rmtree(p)
        subprocess.run(["tar", "-xzf", v2_arc, "-C", lang_dir], check=True)
        continue

    # Remove any unverified or stale v2 archive/audit marker before rebuilding
    if os.path.exists(v2_arc):
        os.remove(v2_arc)
    if os.path.exists(audit_ok_path):
        os.remove(audit_ok_path)

    run1_arc = os.path.join(RUN1_ROOT, f"{lang}_preprocessed.tar.gz")
    if not os.path.exists(run1_arc):
        raise FileNotFoundError(f"Missing Run 1 archive: {run1_arc}")

    print(f"\n==================== [{lang}] Stage R1-A: Alignment-Only Restore & Rebuild ====================")
    lang_stage = os.path.join(STAGING_DIR, lang)
    if os.path.exists(lang_stage):
        shutil.rmtree(lang_stage)
    os.makedirs(lang_stage, exist_ok=True)

    print(f"1. Extracting Run 1 archive {run1_arc} to temporary staging {lang_stage} ...")
    subprocess.run(["tar", "-xzf", run1_arc, "-C", lang_stage], check=True)

    # Clean local preprocessed_data & raw_data in repo so no stale Run 1 features remain
    for clean_sub in ["preprocessed_data", "raw_data"]:
        target_sub = os.path.join(lang_dir, clean_sub)
        if os.path.exists(target_sub):
            shutil.rmtree(target_sub)
    os.makedirs(os.path.join(lang_dir, "preprocessed_data"), exist_ok=True)

    # Copy ONLY TextGrid and lexicon from staging
    staged_tg = os.path.join(lang_stage, "preprocessed_data", "TextGrid")
    if not os.path.exists(staged_tg):
        raise RuntimeError(f"No TextGrid directory found inside {run1_arc}")
    shutil.copytree(staged_tg, os.path.join(lang_dir, "preprocessed_data", "TextGrid"))

    staged_lex = os.path.join(lang_stage, "lexicon")
    if os.path.exists(staged_lex):
        if os.path.exists(os.path.join(lang_dir, "lexicon")):
            shutil.rmtree(os.path.join(lang_dir, "lexicon"))
        shutil.copytree(staged_lex, os.path.join(lang_dir, "lexicon"))

    tg_files = []
    for root, _, files in os.walk(os.path.join(lang_dir, "preprocessed_data", "TextGrid")):
        for f in files:
            if f.endswith(".TextGrid"):
                tg_files.append(os.path.join(root, f))
    print(f"   Restored {len(tg_files)} .TextGrid files (and lexicon) into {lang_dir}. Old mel/pitch/energy/duration discarded.")

    # 2. Regenerate raw waveforms with seed=42
    if lang == "English":
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except Exception:
            if not os.environ.get("HF_TOKEN"):
                os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face Read Token (hf_...) for ai4bharat/Svarah: ")

    print(f"2. Running prepare_data.py --train_hours 5.0 --val_hours 1.0 --seed 42 for {lang} ...")
    subprocess.run(
        [sys.executable, "prepare_data.py", "--train_hours", "5.0", "--val_hours", "1.0", "--seed", "42"],
        cwd=lang_dir,
        check=True,
    )

    # 3. Verify primary and reserve manifests + data_stats.json + TextGrid-to-WAV duration agreement
    for req_split in ["train_filelist.txt", "val_filelist.txt"]:
        old_m = parse_manifest(os.path.join(lang_stage, "raw_data", req_split))
        new_m = parse_manifest(os.path.join(lang_dir, "raw_data", req_split))
        compare_manifests(lang, req_split, old_m, new_m, required=True)

    for res_split in ["reserve_train_filelist.txt", "reserve_val_filelist.txt"]:
        old_m = parse_manifest(os.path.join(lang_stage, "raw_data", res_split))
        new_m = parse_manifest(os.path.join(lang_dir, "raw_data", res_split))
        compare_manifests(lang, res_split, old_m, new_m, required=False)

    old_stats_path = os.path.join(lang_stage, "raw_data", "data_stats.json")
    new_stats_path = os.path.join(lang_dir, "raw_data", "data_stats.json")
    if not os.path.exists(old_stats_path) or not os.path.exists(new_stats_path):
        raise RuntimeError(f"[{lang}] Missing data_stats.json in staging or regenerated raw_data!")
    old_ds = json.load(open(old_stats_path, encoding="utf-8"))
    new_ds = json.load(open(new_stats_path, encoding="utf-8"))
    for k in ["train_utterances", "val_utterances", "reserve_train_utterances", "reserve_val_utterances", "train_speakers", "val_speakers"]:
        if old_ds.get(k) != new_ds.get(k):
            raise RuntimeError(f"[{lang}] data_stats.json mismatch on {k}: Run 1={old_ds.get(k)} vs regenerated={new_ds.get(k)}")
    print(f"   data_stats.json match verified (train={new_ds['train_utterances']}, val={new_ds['val_utterances']}, res_train={new_ds['reserve_train_utterances']}, res_val={new_ds['reserve_val_utterances']}).")

    # Verify every restored TextGrid against its regenerated WAV file
    missing_wavs = 0
    dur_mismatch_wavs = 0
    for tg_path in tg_files:
        spk = os.path.basename(os.path.dirname(tg_path))
        utt_id = os.path.splitext(os.path.basename(tg_path))[0]
        wav_path = os.path.join(lang_dir, "raw_data", spk, f"{utt_id}.wav")
        if not os.path.exists(wav_path):
            missing_wavs += 1
            continue
        sr, wav_arr = wavfile.read(wav_path)
        wav_dur = len(wav_arr) / float(sr)
        tg_obj = tgt.io.read_textgrid(tg_path, include_empty_intervals=True)
        phone_tier = tg_obj.get_tier_by_name("phones")
        if abs(wav_dur - phone_tier.end_time) > 0.05:
            dur_mismatch_wavs += 1
    print(f"   TextGrid-to-WAV timeline check across {len(tg_files)} files: missing_wavs={missing_wavs}, dur_mismatch(>50ms)={dur_mismatch_wavs}")
    if missing_wavs > 0 or dur_mismatch_wavs > 0:
        raise RuntimeError(f"[{lang}] Regenerated WAVs do not match restored TextGrids (missing={missing_wavs}, dur_mismatch={dur_mismatch_wavs})!")

    # 4. Run hardened preprocess.py
    print(f"3. Running hardened preprocess.py --config config.yaml for {lang} ...")
    subprocess.run(
        [sys.executable, "preprocess.py", "--config", "config.yaml"],
        cwd=lang_dir,
        check=True,
    )

    if os.path.exists(lang_stage):
        shutil.rmtree(lang_stage)


In [ ]:
# Cell R1-B: Mandatory Data Integrity, Retained Hours (~5h/1h), Train-Only Stats & Pause Audit -> Archive Validated v2 Features
import os, sys, json, hashlib, subprocess
from datetime import datetime, timezone
import numpy as np

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"

def load_all_manifest_hashes(raw_dir):
    sha_map = {}
    for mf in ["train_filelist.txt", "val_filelist.txt", "reserve_train_filelist.txt", "reserve_val_filelist.txt"]:
        p = os.path.join(raw_dir, mf)
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.rstrip("\r\n").split("|", 4)
                    if len(parts) == 5:
                        sha_map[parts[0]] = parts[3]
    return sha_map

audit_report = {}
for lang in ["English", "Telugu"]:
    lang_dir = os.path.join(REPO_DIR, lang)
    pdir = os.path.join(lang_dir, "preprocessed_data")
    rdir = os.path.join(lang_dir, "raw_data")
    summary_path = os.path.join(pdir, "preprocess_summary.json")
    stats_path = os.path.join(pdir, "stats.json")
    speakers_path = os.path.join(pdir, "speakers.json")

    summary = json.load(open(summary_path, "r", encoding="utf-8")) if os.path.exists(summary_path) else {}
    stats = json.load(open(stats_path, "r", encoding="utf-8"))
    speakers = json.load(open(speakers_path, "r", encoding="utf-8"))
    sha_map = load_all_manifest_hashes(rdir)

    split_info = {}
    train_utts, val_utts = set(), set()
    train_hashes, val_hashes = set(), set()
    train_spks, val_spks = set(), set()
    train_pitch_vals, train_energy_vals = [], []

    for split in ["train", "val"]:
        lines = [l.strip() for l in open(os.path.join(pdir, f"{split}.txt"), encoding="utf-8") if l.strip()]
        len_mismatch = 0
        mel_mismatch = 0
        nonfinite = 0
        utts_with_sp = 0
        total_sp_tokens = 0
        sp_loudness, vowel_loudness = [], []
        phone_loud = {}
        all_loud, all_eng = [], []
        total_frames = 0

        for line in lines:
            utt_id, spk, ph_str, _ = line.split("|")[:4]
            phones = ph_str.strip("{}").split()
            if split == "train":
                train_utts.add(utt_id)
                train_spks.add(spk)
                if utt_id in sha_map:
                    train_hashes.add(sha_map[utt_id])
            else:
                val_utts.add(utt_id)
                val_spks.add(spk)
                if utt_id in sha_map:
                    val_hashes.add(sha_map[utt_id])

            dur = np.load(os.path.join(pdir, "duration", f"{spk}-duration-{utt_id}.npy"))
            pitch = np.load(os.path.join(pdir, "pitch", f"{spk}-pitch-{utt_id}.npy"))
            energy = np.load(os.path.join(pdir, "energy", f"{spk}-energy-{utt_id}.npy"))
            mel = np.load(os.path.join(pdir, "mel", f"{spk}-mel-{utt_id}.npy"))

            if split == "train":
                train_pitch_vals.append(pitch)
                train_energy_vals.append(energy)

            if not (len(phones) == len(dur) == len(pitch) == len(energy)):
                len_mismatch += 1
            if int(np.sum(dur)) != int(mel.shape[0]):
                mel_mismatch += 1
            if not (np.isfinite(dur).all() and np.isfinite(pitch).all() and np.isfinite(energy).all() and np.isfinite(mel).all()):
                nonfinite += 1

            total_frames += int(mel.shape[0])
            if "sp" in phones:
                utts_with_sp += 1
                total_sp_tokens += sum(1 for p in phones if p == "sp")

            idx = 0
            for p, d, e in zip(phones, dur, energy):
                d = int(d)
                if d > 0:
                    loud = float(mel[idx:idx+d].mean())
                    all_loud.append(loud)
                    all_eng.append(float(e))
                    phone_loud.setdefault(p, []).append(loud)
                    if p in ("sp", "sil", "spn"):
                        sp_loudness.append(loud)
                    else:
                        vowel_loudness.append(loud)
                idx += d

        grand_mean = np.mean(all_loud)
        ss_tot = sum((x - grand_mean) ** 2 for x in all_loud)
        ss_bet = sum(len(v) * (np.mean(v) - grand_mean) ** 2 for v in phone_loud.values())
        r2_mel_loud = float(ss_bet / ss_tot) if ss_tot > 0 else 0.0
        corr_eng = float(np.corrcoef(all_eng, all_loud)[0, 1]) if len(all_eng) > 1 else 0.0
        hours = (total_frames * 256.0 / 22050.0) / 3600.0

        split_info[split] = {
            "utts": len(lines),
            "hours": round(hours, 3),
            "len_mismatch": len_mismatch,
            "mel_mismatch": mel_mismatch,
            "nonfinite": nonfinite,
            "utts_with_sp": utts_with_sp,
            "utts_with_sp_pct": round(100.0 * utts_with_sp / max(1, len(lines)), 1),
            "total_sp_tokens": total_sp_tokens,
            "mean_sp_loudness": round(float(np.mean(sp_loudness)), 3) if sp_loudness else None,
            "mean_non_sp_loudness": round(float(np.mean(vowel_loudness)), 3) if vowel_loudness else None,
            "corr_saved_energy_mel_loudness": round(corr_eng, 3),
            "r2_mel_loudness_by_phone": round(r2_mel_loud, 3),
        }

    # Verify train-only normalization (outlier-cleaned mean/std) & quantization bounds (min/max) in stats.json
    def _remove_outlier(vals):
        vals = np.array(vals, dtype=np.float64)
        if len(vals) == 0:
            return vals
        p25, p75 = np.percentile(vals, 25), np.percentile(vals, 75)
        iqr = p75 - p25
        return vals[np.logical_and(vals > (p25 - 1.5 * iqr), vals < (p75 + 1.5 * iqr))]

    cat_p = np.concatenate(train_pitch_vals)
    cat_e = np.concatenate(train_energy_vals)
    p_min_ok = abs(float(np.min(cat_p)) - float(stats["pitch"][0])) < 1e-4
    p_max_ok = abs(float(np.max(cat_p)) - float(stats["pitch"][1])) < 1e-4
    e_min_ok = abs(float(np.min(cat_e)) - float(stats["energy"][0])) < 1e-4
    e_max_ok = abs(float(np.max(cat_e)) - float(stats["energy"][1])) < 1e-4

    clean_norm_p = np.concatenate([_remove_outlier(arr) for arr in train_pitch_vals if len(_remove_outlier(arr)) > 0])
    clean_norm_e = np.concatenate([_remove_outlier(arr) for arr in train_energy_vals if len(_remove_outlier(arr)) > 0])
    scaler_train_only_ok = (
        abs(float(np.mean(clean_norm_p))) < 1e-2
        and abs(float(np.std(clean_norm_p)) - 1.0) < 1e-2
        and abs(float(np.mean(clean_norm_e))) < 1e-2
        and abs(float(np.std(clean_norm_e)) - 1.0) < 1e-2
    )

    utt_overlap = len(train_utts & val_utts)
    sha_overlap = len(train_hashes & val_hashes)
    unseen_val_spks = len(val_spks - train_spks)
    train_h = split_info["train"]["hours"]
    val_h = split_info["val"]["hours"]

    audit_report[lang] = {
        "num_speakers": len(speakers),
        "utt_overlap": utt_overlap,
        "sha1_overlap": sha_overlap,
        "unseen_val_speakers": unseen_val_spks,
        "train_only_bounds_verified": bool(p_min_ok and p_max_ok and e_min_ok and e_max_ok and scaler_train_only_ok),
        "stats_json": stats,
        "preprocess_summary": summary,
        "splits": split_info,
    }

    print(f"\n==================== {lang} R1-B Integrity Audit ====================")
    print(
        f"Speakers: {len(speakers)} | ID Overlap: {utt_overlap} | SHA-1 Overlap: {sha_overlap} | "
        f"Unseen Val Speakers: {unseen_val_spks} | Train-Only Bounds Verified: {audit_report[lang]['train_only_bounds_verified']}"
    )
    for s in ["train", "val"]:
        si = split_info[s]
        print(
            f"  [{s.upper()}] {si['utts']} utts ({si['hours']}h) | "
            f"len_err={si['len_mismatch']} | mel_err={si['mel_mismatch']} | nonfinite={si['nonfinite']} | "
            f"utts_with_sp={si['utts_with_sp']} ({si['utts_with_sp_pct']}%, {si['total_sp_tokens']} tokens) | "
            f"sp_loud={si['mean_sp_loudness']} vs speech_loud={si['mean_non_sp_loudness']} | "
            f"corr={si['corr_saved_energy_mel_loudness']} | R2={si['r2_mel_loudness_by_phone']}"
        )

    # Enforce all hard gates before allowing archive creation
    assert utt_overlap == 0, f"{lang}: Train/Val utterance ID overlap detected!"
    assert sha_overlap == 0, f"{lang}: Train/Val audio SHA-1 overlap detected!"
    assert unseen_val_spks == 0, f"{lang}: Validation contains speakers not in training!"
    assert split_info["train"]["len_mismatch"] == 0 and split_info["val"]["len_mismatch"] == 0
    assert split_info["train"]["mel_mismatch"] == 0 and split_info["val"]["mel_mismatch"] == 0
    assert split_info["train"]["nonfinite"] == 0 and split_info["val"]["nonfinite"] == 0
    assert split_info["train"]["total_sp_tokens"] > 0, f"{lang}: Zero 'sp' pause tokens found!"
    assert audit_report[lang]["train_only_bounds_verified"], f"{lang}: stats.json bounds/scaler do not match train.txt!"
    assert 4.5 <= train_h <= 5.5, f"{lang}: Retained training hours ({train_h}h) outside target ~5.0h [4.5h, 5.5h]!"
    assert 0.85 <= val_h <= 1.2, f"{lang}: Retained validation hours ({val_h}h) outside target ~1.0h [0.85h, 1.2h]!"

    # Archive validated v2 features only after all assertions pass
    v2_arc = os.path.join(RUN2_ROOT, f"{lang}_preprocessed_v2.tar.gz")
    if not os.path.exists(v2_arc):
        print(f"  Archiving validated {lang} v2 features to {v2_arc} ...")
        files_to_archive = [
            "preprocessed_data",
            "lexicon",
            "raw_data/data_stats.json",
            "raw_data/train_filelist.txt",
            "raw_data/val_filelist.txt",
            "raw_data/reserve_train_filelist.txt",
            "raw_data/reserve_val_filelist.txt",
        ]
        subprocess.run(["tar", "-czf", v2_arc, "-C", lang_dir] + files_to_archive, check=True)
    arc_sha256 = hashlib.sha256(open(v2_arc, "rb").read()).hexdigest()
    audit_report[lang]["v2_archive"] = {
        "path": v2_arc,
        "size_mb": round(os.path.getsize(v2_arc) / (1024 * 1024), 2),
        "sha256": arc_sha256,
    }
    print(f"  Verified Run 2 archive: {v2_arc} ({audit_report[lang]['v2_archive']['size_mb']} MB, sha256={arc_sha256[:16]}...)")

audit_out = os.path.join(RUN2_ROOT, "diagnostics", "r1_integrity_audit.json")
with open(audit_out, "w", encoding="utf-8") as f:
    json.dump(audit_report, f, indent=2)
print(f"\nSUCCESS: R1-B Data Integrity, Split Hours (~5h/1h), and Train-Only Bounds Audit passed! Saved to {audit_out}")


## Section R2: Isolated Vocoder & Frontend Quality Gate
- Runs in **isolated per-language Python subprocesses** so `sys.modules` never mixes `English` and `Telugu` modules.
- Reconstructs multiple clips for English and both Telugu voices (`te_female`, `te_male`) from rebuilt ground-truth mels via Universal HiFi-GAN, compares against the original 22.05 kHz WAVs, saves paired files in `runs/pause_fix_v2/diagnostics/vocoder_gate/`, and displays interactive `<audio>` players.
- Verifies all 10 required test sentences (`ENGLISH_TEST_SENTENCES` `E1–E4` and `TELUGU_TEST_SENTENCES` `T1–T4`) through the exact language frontends.

In [ ]:
import os, sys, json, subprocess
from IPython.display import Audio, display

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
VOC_OUT_DIR = os.path.join(RUN2_ROOT, "diagnostics", "vocoder_gate")
os.makedirs(VOC_OUT_DIR, exist_ok=True)

probe_script = r'''
import os, sys, json, yaml, torch, shutil
import numpy as np
from scipy.io import wavfile

lang = sys.argv[1]
lang_dir = sys.argv[2]
out_dir = sys.argv[3]
sys.path.insert(0, lang_dir)

from utils.model import get_vocoder, vocoder_infer
from text.symbols import symbols
if lang == "English":
    from inference import ENGLISH_TEST_SENTENCES as test_sentences_dict, preprocess_english as preprocess_fn
else:
    from inference import TELUGU_TEST_SENTENCES as test_sentences_dict, preprocess_telugu as preprocess_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open(os.path.join(lang_dir, "config.yaml"), encoding="utf-8"))
vocoder = get_vocoder(cfg, device)

pdir = os.path.join(lang_dir, "preprocessed_data")
rdir = os.path.join(lang_dir, "raw_data")
lines = [l.strip().split("|") for l in open(os.path.join(pdir, "val.txt"), encoding="utf-8") if l.strip()]

# Pick 3 clips covering distinct speakers (including both te_female and te_male for Telugu)
selected = []
seen_spk = set()
for parts in lines:
    utt_id, spk = parts[0], parts[1]
    if spk not in seen_spk:
        selected.append((utt_id, spk, parts[3]))
        seen_spk.add(spk)
    if len(selected) >= 3:
        break
for parts in lines:
    if len(selected) >= 3:
        break
    item = (parts[0], parts[1], parts[3])
    if item not in selected:
        selected.append(item)

records = []
for utt_id, spk, raw_txt in selected[:3]:
    mel_path = os.path.join(pdir, "mel", f"{spk}-mel-{utt_id}.npy")
    mel = np.load(mel_path)
    mel_tensor = torch.from_numpy(mel).float().unsqueeze(0).transpose(1, 2).to(device)
    wav_rec = vocoder_infer(mel_tensor, vocoder, cfg, cfg)[0]
    assert np.isfinite(wav_rec).all() and np.max(np.abs(wav_rec)) > 100, f"Degenerate vocoder output for {utt_id}"
    rec_path = os.path.join(out_dir, f"{lang}_{spk}_{utt_id}_reconstructed.wav")
    wavfile.write(rec_path, 22050, wav_rec)

    orig_src = os.path.join(rdir, spk, f"{utt_id}.wav")
    orig_dst = os.path.join(out_dir, f"{lang}_{spk}_{utt_id}_orig.wav")
    orig_dur_s = None
    if os.path.exists(orig_src):
        shutil.copy2(orig_src, orig_dst)
    if os.path.exists(orig_dst):
        sr_o, wav_o = wavfile.read(orig_dst)
        orig_dur_s = round(len(wav_o) / float(sr_o), 2)
    records.append({
        "utt_id": utt_id,
        "speaker": spk,
        "text": raw_txt,
        "mel_frames": int(mel.shape[0]),
        "orig_duration_s": orig_dur_s,
        "rec_duration_s": round(len(wav_rec) / 22050.0, 2),
        "peak_norm": round(float(np.max(np.abs(wav_rec))) / 32768.0, 3),
        "orig_wav": orig_dst if os.path.exists(orig_dst) else None,
        "rec_wav": rec_path,
    })

# Verify all 5 test sentences for this language through the exact frontend
symbol_set = set(symbols)
frontend_checks = []
for sid, raw_text in test_sentences_dict.items():
    seq, phones, norm_text = preprocess_fn(raw_text, cfg)
    unknown = [p for p in phones if f"@{p}" not in symbol_set and p not in symbol_set]
    assert len(seq) == len(phones) and len(seq) > 0 and not unknown, (
        f"Frontend check failed for {sid}: len(seq)={len(seq)}, len(phones)={len(phones)}, unknown={unknown}"
    )
    frontend_checks.append({
        "id": sid,
        "raw_text": raw_text,
        "normalized_text": norm_text,
        "num_phones": len(phones),
        "phones_preview": " ".join(phones[:25]) + (" ..." if len(phones) > 25 else ""),
    })

out_json = os.path.join(out_dir, f"{lang}_r2_gate.json")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump({"vocoder_samples": records, "frontend_checks": frontend_checks}, f, indent=2, ensure_ascii=False)
'''

with open("/tmp/run_r2_gate.py", "w", encoding="utf-8") as f:
    f.write(probe_script)

for lang in ["English", "Telugu"]:
    print(f"\n==================== Running Isolated R2 Gate Subprocess for {lang} ====================")
    subprocess.run([sys.executable, "/tmp/run_r2_gate.py", lang, os.path.join(REPO_DIR, lang), VOC_OUT_DIR], check=True)
    res = json.load(open(os.path.join(VOC_OUT_DIR, f"{lang}_r2_gate.json"), encoding="utf-8"))

    print(f"\n--- {lang} Frontend Check on Mandatory Test Sentences ---")
    for fc in res["frontend_checks"]:
        print(f"  [{fc['id']}] ({fc['num_phones']} phones) raw={fc['raw_text']!r}")
        print(f"       norm={fc['normalized_text']!r}")
        print(f"       phones={fc['phones_preview']}")

    print(f"\n--- {lang} Ground-Truth Mel -> Universal HiFi-GAN Reconstruction Samples ---")
    for rec in res["vocoder_samples"]:
        print(
            f"\nClip: {rec['utt_id']} | Speaker: {rec['speaker']} | "
            f"orig={rec['orig_duration_s']}s vs rec={rec['rec_duration_s']}s (peak={rec['peak_norm']}) | "
            f"Text: {rec['text'][:90]}"
        )
        if rec["orig_wav"] and os.path.exists(rec["orig_wav"]):
            print("  Original 22.05kHz WAV:")
            display(Audio(rec["orig_wav"]))
        print("  HiFi-GAN Reconstructed WAV from Rebuilt Mel:")
        display(Audio(rec["rec_wav"]))


## Section R3: Small Subset Overfit (64 Clips, 2,000 Steps) — Reference-Conditioned vs. Free-Running Speech Gate
- Trains a fast **2,000-step diagnostic overfit** on a 64-clip subset per language in a temporary directory (`/content/overfit64_work`).
- Evaluates **both**:
  1. **Reference-conditioned reconstruction** (ground-truth duration, pitch, and energy targets passed into Variance Adaptor), and
  2. **Free-running inference** (predicted duration, predicted pitch, and predicted energy only)
  on seen training clips and held-out validation clips, saving all WAVs to `runs/pause_fix_v2/diagnostics/overfit64/`.

In [ ]:
import os, sys, json, glob, shutil, subprocess
from IPython.display import Audio, display

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
OVERFIT_ROOT = "/content/overfit64_work"
OVERFIT_DIAG_DIR = os.path.join(RUN2_ROOT, "diagnostics", "overfit64")
os.makedirs(OVERFIT_DIAG_DIR, exist_ok=True)

OVERFIT_STEPS = 2000

overfit_eval_script = r'''
import os, sys, json, yaml, torch
import numpy as np
from scipy.io import wavfile

lang = sys.argv[1]
lang_dir = sys.argv[2]
prep_dir = sys.argv[3]
ckpt_dir = sys.argv[4]
out_dir = sys.argv[5]
step = int(sys.argv[6])

sys.path.insert(0, lang_dir)
from utils.model import get_model, get_vocoder, vocoder_infer
from text import text_to_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open(os.path.join(lang_dir, "config.yaml"), encoding="utf-8"))
cfg["path"]["preprocessed_path"] = prep_dir
cfg["path"]["ckpt_path"] = ckpt_dir
configs = (cfg, cfg, cfg)

class DummyArgs:
    restore_step = step
    ckpt_file = os.path.join(ckpt_dir, f"{step}.pth.tar")

model = get_model(DummyArgs(), configs, device, train=False)
vocoder = get_vocoder(cfg, device)
speakers = json.load(open(os.path.join(prep_dir, "speakers.json"), encoding="utf-8"))
cleaners = cfg["preprocessing"]["text"]["text_cleaners"]

train_lines = [l.strip().split("|") for l in open(os.path.join(prep_dir, "train.txt"), encoding="utf-8") if l.strip()]
val_lines = [l.strip().split("|") for l in open(os.path.join(lang_dir, "preprocessed_data", "val.txt"), encoding="utf-8") if l.strip()]

samples_to_synth = [
    ("seen_train_1", train_lines[0]),
    ("seen_train_2", train_lines[min(1, len(train_lines)-1)]),
    ("heldout_val_1", val_lines[0]),
]

results = []
for tag, parts in samples_to_synth:
    utt_id, spk_name, ph_str, raw_txt = parts[0], parts[1], parts[2], parts[3]
    spk_id = speakers.get(spk_name, 0)
    seq = np.array(text_to_sequence(ph_str, cleaners), dtype=np.int64)
    src = torch.from_numpy(seq).long().unsqueeze(0).to(device)
    src_len = torch.tensor([len(seq)], dtype=torch.long, device=device)
    spk_tensor = torch.tensor([spk_id], dtype=torch.long, device=device)

    # Load ground-truth targets for reference-conditioned comparison
    gt_dur = np.load(os.path.join(prep_dir, "duration", f"{spk_name}-duration-{utt_id}.npy"))
    gt_pitch = np.load(os.path.join(prep_dir, "pitch", f"{spk_name}-pitch-{utt_id}.npy"))
    gt_energy = np.load(os.path.join(prep_dir, "energy", f"{spk_name}-energy-{utt_id}.npy"))
    d_t = torch.from_numpy(gt_dur).long().unsqueeze(0).to(device)
    p_t = torch.from_numpy(gt_pitch).float().unsqueeze(0).to(device)
    e_t = torch.from_numpy(gt_energy).float().unsqueeze(0).to(device)
    mel_len_gt = int(np.sum(gt_dur))
    mel_lens_t = torch.tensor([mel_len_gt], dtype=torch.long, device=device)

    with torch.no_grad():
        # 1. Reference-conditioned forward pass
        out_ref = model(
            spk_tensor, src, src_len, int(len(seq)),
            mel_lens=mel_lens_t, max_mel_len=mel_len_gt,
            p_targets=p_t, e_targets=e_t, d_targets=d_t,
        )
        mel_ref = out_ref[1].transpose(1, 2)
        wav_ref = vocoder_infer(mel_ref, vocoder, cfg, cfg)[0]

        # 2. Free-running forward pass (predicted duration, pitch, energy)
        out_free = model(spk_tensor, src, src_len, int(len(seq)))
        mel_free = out_free[1].transpose(1, 2)
        dur_pred = out_free[5][0].detach().cpu().numpy()
        wav_free = vocoder_infer(mel_free, vocoder, cfg, cfg)[0]

    ref_path = os.path.join(out_dir, f"{lang}_overfit_{tag}_{utt_id}_ref_conditioned.wav")
    free_path = os.path.join(out_dir, f"{lang}_overfit_{tag}_{utt_id}_free_running.wav")
    wavfile.write(ref_path, 22050, wav_ref)
    wavfile.write(free_path, 22050, wav_free)
    results.append({
        "tag": tag,
        "utt_id": utt_id,
        "speaker": spk_name,
        "text": raw_txt,
        "gt_frames": mel_len_gt,
        "pred_frames": int(np.sum(dur_pred)),
        "ref_duration_s": round(len(wav_ref) / 22050.0, 2),
        "free_duration_s": round(len(wav_free) / 22050.0, 2),
        "ref_wav_path": ref_path,
        "free_wav_path": free_path,
    })

with open(os.path.join(out_dir, f"{lang}_overfit_summary.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
'''
with open("/tmp/eval_overfit.py", "w", encoding="utf-8") as f:
    f.write(overfit_eval_script)

for lang in ["English", "Telugu"]:
    print(f"\n==================== [{lang}] Stage R3: 64-Clip Overfit Diagnostic ({OVERFIT_STEPS} steps) ====================")
    lang_dir = os.path.join(REPO_DIR, lang)
    orig_prep = os.path.join(lang_dir, "preprocessed_data")
    work_dir = os.path.join(OVERFIT_ROOT, lang)
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)
    prep_sub = os.path.join(work_dir, "preprocessed_data")
    ckpt_sub = os.path.join(work_dir, "checkpoints")
    log_sub = os.path.join(work_dir, "logs")
    res_sub = os.path.join(work_dir, "results")
    os.makedirs(prep_sub, exist_ok=True)

    for item in ["mel", "pitch", "energy", "duration", "stats.json", "speakers.json"]:
        os.symlink(os.path.join(orig_prep, item), os.path.join(prep_sub, item))

    all_train = [l for l in open(os.path.join(orig_prep, "train.txt"), encoding="utf-8") if l.strip()]
    spk_counts = {}
    for l in all_train:
        spk = l.split("|")[1]
        spk_counts[spk] = spk_counts.get(spk, 0) + 1
    top_spk = max(spk_counts, key=spk_counts.get)
    top_spk_lines = [l for l in all_train if l.split("|")[1] == top_spk]
    subset64 = (top_spk_lines + [l for l in all_train if l.split("|")[1] != top_spk])[:64]

    with open(os.path.join(prep_sub, "train.txt"), "w", encoding="utf-8") as f:
        f.writelines(subset64)
    with open(os.path.join(prep_sub, "val.txt"), "w", encoding="utf-8") as f:
        f.writelines(subset64[:16])

    subprocess.run([
        sys.executable, "train.py",
        "--config", "config.yaml",
        "--restore_step", "0",
        "--total_step", str(OVERFIT_STEPS),
        "--val_step", str(OVERFIT_STEPS),
        "--save_step", str(OVERFIT_STEPS),
        "--synth_step", str(OVERFIT_STEPS),
        "--preprocessed_path", prep_sub,
        "--ckpt_path", ckpt_sub,
        "--log_path", log_sub,
        "--result_path", res_sub,
    ], cwd=lang_dir, check=True)

    subprocess.run([
        sys.executable, "/tmp/eval_overfit.py",
        lang, lang_dir, prep_sub, ckpt_sub, OVERFIT_DIAG_DIR, str(OVERFIT_STEPS)
    ], check=True)

    summary = json.load(open(os.path.join(OVERFIT_DIAG_DIR, f"{lang}_overfit_summary.json"), encoding="utf-8"))
    for item in summary:
        print(
            f"\n[{lang} Overfit {item['tag']}] {item['utt_id']} ({item['speaker']}) | "
            f"GT={item['gt_frames']}f ({item['ref_duration_s']}s) vs Pred={item['pred_frames']}f ({item['free_duration_s']}s) | "
            f"Text: {item['text'][:90]}"
        )
        print("  1. Reference-Conditioned (GT duration/pitch/energy):")
        display(Audio(item["ref_wav_path"]))
        print("  2. Free-Running (Predicted duration/pitch/energy):")
        display(Audio(item["free_wav_path"]))


## Section R4: Held-Out Pilot (`5,000` Steps Default) & Full Independent Production Runs in `runs/pause_fix_v2`
- **Staged Pilot Gate (`PILOT_OR_TOTAL_STEP = 5000` by default):**
  1. First run **R4-A** and **R4-B** with the default `ALLOW_FULL_PRODUCTION_RUN = False` (`5,000`-step full-data pilot, validating and saving at `1,000`, `3,000`, and `5,000` steps).
  2. Run **R4-C** to inspect the held-out validation loss table and listen to **free-running held-out speech** at steps `1,000`, `3,000`, `5,000`, and `best_val`.
  3. Only after confirming audible improvement in **R4-C**, set `ALLOW_FULL_PRODUCTION_RUN = True` (and `FULL_TARGET_STEP = 60000`) to resume seamlessly from step `5,000`.

In [ ]:
# R4-A: Train English FastSpeech 2 Model (Defaults to 5,000-step Pilot in runs/pause_fix_v2/English)
import os, sys, glob, subprocess

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
EN_DIR = os.path.join(REPO_DIR, "English")
EN_CKPT_DIR = os.path.join(RUN2_ROOT, "English", "checkpoints")
EN_LOG_DIR = os.path.join(RUN2_ROOT, "English", "logs")
EN_RES_DIR = os.path.join(RUN2_ROOT, "Results", "English")

os.makedirs(EN_CKPT_DIR, exist_ok=True)
os.makedirs(EN_LOG_DIR, exist_ok=True)
os.makedirs(EN_RES_DIR, exist_ok=True)

# Require R1-B audit to have passed before starting R4
audit_file = os.path.join(RUN2_ROOT, "diagnostics", "r1_integrity_audit.json")
assert os.path.exists(audit_file), f"Run Cell R1-B first! Missing {audit_file}"

if not os.path.exists(os.path.join(EN_DIR, "preprocessed_data", "train.txt")):
    en_v2_arc = os.path.join(RUN2_ROOT, "English_preprocessed_v2.tar.gz")
    assert os.path.exists(en_v2_arc), f"Missing {en_v2_arc}"
    subprocess.run(["tar", "-xzf", en_v2_arc, "-C", EN_DIR], check=True)

# Gate: Keep ALLOW_FULL_PRODUCTION_RUN = False for the initial 5,000-step pilot.
# Set ALLOW_FULL_PRODUCTION_RUN = True only after reviewing R1-R3 and the 5k pilot in Cell R4-C.
ALLOW_FULL_PRODUCTION_RUN = False
PILOT_STEP = 5000
FULL_TARGET_STEP = 60000
target_step = FULL_TARGET_STEP if ALLOW_FULL_PRODUCTION_RUN else PILOT_STEP
save_step = 5000 if ALLOW_FULL_PRODUCTION_RUN else 1000

ckpts = [
    int(os.path.basename(f).replace(".pth.tar", ""))
    for f in glob.glob(os.path.join(EN_CKPT_DIR, "*.pth.tar"))
    if os.path.basename(f).replace(".pth.tar", "").isdigit()
]
restore_step = max(ckpts) if ckpts else 0
print(f"[English Run 2] Starting/Resuming from step {restore_step} -> {target_step} (save_step={save_step}) in {EN_CKPT_DIR} ...")

if restore_step < target_step:
    subprocess.run([
        sys.executable, "train.py",
        "--config", "config.yaml",
        "--restore_step", str(restore_step),
        "--total_step", str(target_step),
        "--val_step", "1000",
        "--save_step", str(save_step),
        "--ckpt_path", EN_CKPT_DIR,
        "--log_path", EN_LOG_DIR,
        "--result_path", EN_RES_DIR,
    ], cwd=EN_DIR, check=True)
else:
    print(f"[English Run 2] Already reached step {restore_step} >= {target_step}. Run Cell R4-C to evaluate checkpoints, or set ALLOW_FULL_PRODUCTION_RUN = True to continue.")


In [ ]:
# R4-B: Train Telugu FastSpeech 2 Model (Defaults to 5,000-step Pilot in runs/pause_fix_v2/Telugu)
import os, sys, glob, subprocess

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
TE_DIR = os.path.join(REPO_DIR, "Telugu")
TE_CKPT_DIR = os.path.join(RUN2_ROOT, "Telugu", "checkpoints")
TE_LOG_DIR = os.path.join(RUN2_ROOT, "Telugu", "logs")
TE_RES_DIR = os.path.join(RUN2_ROOT, "Results", "Telugu")

os.makedirs(TE_CKPT_DIR, exist_ok=True)
os.makedirs(TE_LOG_DIR, exist_ok=True)
os.makedirs(TE_RES_DIR, exist_ok=True)

audit_file = os.path.join(RUN2_ROOT, "diagnostics", "r1_integrity_audit.json")
assert os.path.exists(audit_file), f"Run Cell R1-B first! Missing {audit_file}"

if not os.path.exists(os.path.join(TE_DIR, "preprocessed_data", "train.txt")):
    te_v2_arc = os.path.join(RUN2_ROOT, "Telugu_preprocessed_v2.tar.gz")
    assert os.path.exists(te_v2_arc), f"Missing {te_v2_arc}"
    subprocess.run(["tar", "-xzf", te_v2_arc, "-C", TE_DIR], check=True)

# Gate: Keep ALLOW_FULL_PRODUCTION_RUN = False for the initial 5,000-step pilot.
# Set ALLOW_FULL_PRODUCTION_RUN = True only after reviewing R1-R3 and the 5k pilot in Cell R4-C.
ALLOW_FULL_PRODUCTION_RUN = False
PILOT_STEP = 5000
FULL_TARGET_STEP = 60000
target_step = FULL_TARGET_STEP if ALLOW_FULL_PRODUCTION_RUN else PILOT_STEP
save_step = 5000 if ALLOW_FULL_PRODUCTION_RUN else 1000

ckpts = [
    int(os.path.basename(f).replace(".pth.tar", ""))
    for f in glob.glob(os.path.join(TE_CKPT_DIR, "*.pth.tar"))
    if os.path.basename(f).replace(".pth.tar", "").isdigit()
]
restore_step = max(ckpts) if ckpts else 0
print(f"[Telugu Run 2] Starting/Resuming from step {restore_step} -> {target_step} (save_step={save_step}) in {TE_CKPT_DIR} ...")

if restore_step < target_step:
    subprocess.run([
        sys.executable, "train.py",
        "--config", "config.yaml",
        "--restore_step", str(restore_step),
        "--total_step", str(target_step),
        "--val_step", "1000",
        "--save_step", str(save_step),
        "--ckpt_path", TE_CKPT_DIR,
        "--log_path", TE_LOG_DIR,
        "--result_path", TE_RES_DIR,
    ], cwd=TE_DIR, check=True)
else:
    print(f"[Telugu Run 2] Already reached step {restore_step} >= {target_step}. Run Cell R4-C to evaluate checkpoints, or set ALLOW_FULL_PRODUCTION_RUN = True to continue.")


In [ ]:
# R4-C: Checkpoint Validation Loss Table & Free-Running Audio Sweep (Pilot & Production Checkpoints)
import os, sys, csv, json, glob, subprocess
from IPython.display import Audio, display

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
SWEEP_DIR = os.path.join(RUN2_ROOT, "diagnostics", "ckpt_sweep")
os.makedirs(SWEEP_DIR, exist_ok=True)

sweep_script = r'''
import os, sys, json, glob, yaml, torch
import numpy as np
from scipy.io import wavfile

lang = sys.argv[1]
lang_dir = sys.argv[2]
ckpt_dir = sys.argv[3]
out_dir = sys.argv[4]

sys.path.insert(0, lang_dir)
from utils.model import get_model, get_vocoder, vocoder_infer
from text import text_to_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open(os.path.join(lang_dir, "config.yaml"), encoding="utf-8"))
cfg["path"]["ckpt_path"] = ckpt_dir
configs = (cfg, cfg, cfg)
vocoder = get_vocoder(cfg, device)

pdir = os.path.join(lang_dir, "preprocessed_data")
speakers = json.load(open(os.path.join(pdir, "speakers.json"), encoding="utf-8"))
cleaners = cfg["preprocessing"]["text"]["text_cleaners"]
val_lines = [l.strip().split("|") for l in open(os.path.join(pdir, "val.txt"), encoding="utf-8") if l.strip()]

# Pick 2 representative held-out validation sentences
heldout = val_lines[:2]

# Candidate checkpoints to compare (covering both 1k/3k/5k pilot steps and later production steps)
all_ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.pth.tar")))
candidates = []
for name in [
    "1000.pth.tar", "3000.pth.tar", "5000.pth.tar",
    "10000.pth.tar", "20000.pth.tar", "30000.pth.tar",
    "40000.pth.tar", "50000.pth.tar", "60000.pth.tar",
    "best_val.pth.tar",
]:
    p = os.path.join(ckpt_dir, name)
    if os.path.exists(p):
        candidates.append(p)
if not candidates and all_ckpts:
    candidates = all_ckpts[-3:]

records = []
for ckpt_path in candidates:
    cname = os.path.basename(ckpt_path)
    raw_ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    step_num = raw_ckpt.get("step", cname)
    val_loss = raw_ckpt.get("val_loss", None)

    class DummyArgs:
        restore_step = 0
        ckpt_file = ckpt_path

    model = get_model(DummyArgs(), configs, device, train=False)
    sample_wavs = []
    for parts in heldout:
        utt_id, spk_name, ph_str, raw_txt = parts[0], parts[1], parts[2], parts[3]
        spk_id = speakers.get(spk_name, 0)
        seq = np.array(text_to_sequence(ph_str, cleaners), dtype=np.int64)
        src = torch.from_numpy(seq).long().unsqueeze(0).to(device)
        src_len = torch.tensor([len(seq)], dtype=torch.long, device=device)
        spk_tensor = torch.tensor([spk_id], dtype=torch.long, device=device)
        with torch.no_grad():
            out = model(spk_tensor, src, src_len, max_src_len=int(len(seq)))
            mel_post = out[1].transpose(1, 2)
            dur_pred = out[5][0].detach().cpu().numpy()
            wav = vocoder_infer(mel_post, vocoder, cfg, cfg)[0]
        wpath = os.path.join(out_dir, f"{lang}_{cname.replace('.pth.tar','')}_{utt_id}.wav")
        wavfile.write(wpath, 22050, wav)
        sample_wavs.append({
            "utt_id": utt_id,
            "speaker": spk_name,
            "text": raw_txt,
            "pred_frames": int(np.sum(dur_pred)),
            "wav": wpath,
            "dur_s": round(len(wav) / 22050.0, 2),
        })

    records.append({"checkpoint": cname, "step": step_num, "val_loss": val_loss, "samples": sample_wavs})

with open(os.path.join(out_dir, f"{lang}_sweep.json"), "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)
'''
with open("/tmp/run_ckpt_sweep.py", "w", encoding="utf-8") as f:
    f.write(sweep_script)

for lang in ["English", "Telugu"]:
    print(f"\n==================== {lang} Run 2 Validation History & Checkpoint Sweep ====================")
    val_csv = os.path.join(RUN2_ROOT, lang, "logs", "val_losses.csv")
    if os.path.exists(val_csv):
        with open(val_csv, "r", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        print(f"Total validation evaluations logged: {len(rows)}")
        for r in rows:
            print(f"  step={r['step']:>6} | total={float(r['total_loss']):.4f} | mel={float(r['mel_loss']):.4f} | post={float(r['mel_postnet_loss']):.4f} | pitch={float(r['pitch_loss']):.4f} | energy={float(r['energy_loss']):.4f} | dur={float(r['duration_loss']):.4f}")

    ckpt_dir = os.path.join(RUN2_ROOT, lang, "checkpoints")
    if glob.glob(os.path.join(ckpt_dir, "*.pth.tar")):
        subprocess.run([sys.executable, "/tmp/run_ckpt_sweep.py", lang, os.path.join(REPO_DIR, lang), ckpt_dir, SWEEP_DIR], check=True)
        sweep = json.load(open(os.path.join(SWEEP_DIR, f"{lang}_sweep.json"), encoding="utf-8"))
        for rec in sweep:
            print(f"\n--- {lang} Checkpoint: {rec['checkpoint']} (step={rec['step']}, val_loss={rec['val_loss']}) ---")
            for s in rec["samples"]:
                print(f"  Held-out {s['utt_id']} ({s['speaker']}, {s['dur_s']}s, {s['pred_frames']}f): {s['text'][:90]}")
                display(Audio(s["wav"]))


## Section R5: Synthesize All 10 Mandatory Test Sentences (`E1–E4`, `T1–T4`), Generate 30 Variance Plots, Listen & Package Deliverables
- Synthesizes the exact 10 test sentences using **predicted duration, predicted pitch, and predicted energy only**.
- Reads `variance_predictions_summary.json` written by `inference.py`, verifies all 10 WAVs, 30 variance adaptor plots, and 4 loss curves (`training_loss.png`, `validation_loss.png`), displays audio players for listening review, and packages `Results_Deliverables_v2.zip`.

In [ ]:
import os, sys, json, glob, subprocess
from IPython.display import Audio, display

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"

# Set SELECTED_EN_CKPT / SELECTED_TE_CKPT after listening in R4-C (e.g. os.path.join(RUN2_ROOT, "English/checkpoints/best_val.pth.tar")).
# If left None, best_val.pth.tar is used if present, otherwise the latest numeric checkpoint.
SELECTED_EN_CKPT = None
SELECTED_TE_CKPT = None

# Speaker selection: -1 auto-selects highest-count training profile for English; 0 selects te_female (or 1 for te_male) for Telugu
EN_SPEAKER_ID = -1
TE_SPEAKER_ID = 0

for lang, sel_ckpt, spk_id in [
    ("English", SELECTED_EN_CKPT, EN_SPEAKER_ID),
    ("Telugu", SELECTED_TE_CKPT, TE_SPEAKER_ID),
]:
    lang_dir = os.path.join(REPO_DIR, lang)
    ckpt_dir = os.path.join(RUN2_ROOT, lang, "checkpoints")
    res_dir = os.path.join(RUN2_ROOT, "Results", lang)
    os.makedirs(res_dir, exist_ok=True)

    if sel_ckpt is None:
        best_candidate = os.path.join(ckpt_dir, "best_val.pth.tar")
        if os.path.exists(best_candidate):
            sel_ckpt = best_candidate

    cmd = [
        sys.executable, "inference.py",
        "--config", "config.yaml",
        "--speaker_id", str(spk_id),
        "--ckpt_path", ckpt_dir,
        "--result_path", res_dir,
    ]
    if sel_ckpt:
        cmd.extend(["--ckpt_file", sel_ckpt])

    print(f"\n==================== Running Final R5 Inference for {lang} (ckpt={sel_ckpt or 'latest'}) ====================")
    subprocess.run(cmd, cwd=lang_dir, check=True)

    summary_path = os.path.join(res_dir, "variance_predictions_summary.json")
    assert os.path.exists(summary_path), f"Missing {summary_path}!"
    summary = json.load(open(summary_path, encoding="utf-8"))
    for sample_id, item in summary.items():
        dur_sec = item["total_frames"] * 256.0 / 22050.0
        print(
            f"\n[{lang} - {item['id']}] Speaker ID: {item['speaker_id']} | "
            f"Checkpoint Step: {item['checkpoint_step']} | Duration: {dur_sec:.2f}s ({item['total_frames']} frames)"
        )
        print(f"  Raw Input : {item['raw_text']}")
        print(f"  Normalized: {item['normalized_text']}")
        print(f"  Tokens ({len(item['tokens'])}): {' '.join(item['tokens'][:30])}{' ...' if len(item['tokens']) > 30 else ''}")
        wav_file = os.path.join(res_dir, "test_generated_audios", f"{item['id']}.wav")
        display(Audio(wav_file))

# Verify completeness of all deliverables in RUN2_ROOT/Results
for lang in ["English", "Telugu"]:
    rdir = os.path.join(RUN2_ROOT, "Results", lang)
    wavs = glob.glob(os.path.join(rdir, "test_generated_audios", "*.wav"))
    durs = glob.glob(os.path.join(rdir, "variance_adaptor_plots", "duration", "*.png"))
    pitches = glob.glob(os.path.join(rdir, "variance_adaptor_plots", "pitch", "*.png"))
    energies = glob.glob(os.path.join(rdir, "variance_adaptor_plots", "energy", "*.png"))
    t_loss = os.path.exists(os.path.join(rdir, "training_loss.png"))
    v_loss = os.path.exists(os.path.join(rdir, "validation_loss.png"))
    print(f"\n[{lang} Deliverable Check] WAVs={len(wavs)}/5 | Duration Plots={len(durs)}/5 | Pitch Plots={len(pitches)}/5 | Energy Plots={len(energies)}/5 | Loss Curves={t_loss and v_loss}")
    assert len(wavs) == 5 and len(durs) == 5 and len(pitches) == 5 and len(energies) == 5, f"Incomplete test deliverables for {lang}!"
    assert t_loss and v_loss, f"Missing training_loss.png or validation_loss.png for {lang}!"

# Package Results_Deliverables_v2.zip (including Results, diagnostics, and logs)
zip_path = os.path.join(RUN2_ROOT, "Results_Deliverables_v2.zip")
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(
    ["zip", "-r", zip_path, "Results", "diagnostics", "English/logs", "Telugu/logs"],
    cwd=RUN2_ROOT,
    check=True,
)
print(f"\nSUCCESS: Packaged all Run 2 deliverables to {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)!")
